In [6]:
import os
import math
import concurrent.futures
from io import BytesIO
import requests
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from PIL import Image
from tqdm import tqdm

# --- Configuration ---
# Define bounding box extent: (lat, lon)
TOP_LEFT = (33.907267674195495, -118.37438955894808)     # Example: San Francisco area
BOTTOM_RIGHT = (33.875205419908916, -118.28886560981613)

OUTPUT_PARQUET = "extent_tiles.parquet"
MAX_DOWNLOAD_THREADS = 40
TILE_SIZE_METERS = 100.0          # 200m x 200m spatial coverage
IMAGE_PIXEL_DIM = 224             # 224x224 input suitable for CLIP models

# --- Coordinate Helpers ---
def wgs84_to_web_mercator(lon, lat):
    lon_rad = np.radians(lon)
    x = 6378137.0 * lon_rad
    y = 6378137.0 * np.log(np.tan(np.pi / 4.0 + np.radians(lat) / 2.0))
    return x, y

def generate_extent_grid(top_left, bottom_right, step_meters=200.0):
    """Generates a grid of (center_lon, center_lat) points spaced by step_meters."""
    max_lat, min_lon = top_left
    min_lat, max_lon = bottom_right

    mx_min, my_max = wgs84_to_web_mercator(min_lon, max_lat)
    mx_max, my_min = wgs84_to_web_mercator(max_lon, min_lat)

    # Shift centers by half step to cover full extent nicely
    x_coords = np.arange(mx_min + step_meters / 2, mx_max, step_meters)
    y_coords = np.arange(my_min + step_meters / 2, my_max, step_meters)

    # Web Mercator back to WGS84
    points = []
    for mx in x_coords:
        for my in y_coords:
            lon = np.degrees(mx / 6378137.0)
            lat = np.degrees(2.0 * np.arctan(np.exp(my / 6378137.0)) - np.pi / 2.0)
            points.append((lon, lat))
            
    return points

def download_point_image(lon, lat, span_meters=200.0, pixel_dim=224):
    service_url = "https://services.arcgisonline.com/arcgis/rest/services/World_Imagery/MapServer/export"
    try:
        mx, my = wgs84_to_web_mercator(lon, lat)
        half_span = span_meters / 2.0

        mx_min, mx_max = mx - half_span, mx + half_span
        my_min, my_max = my - half_span, my + half_span

        export_params = {
            "bbox": f"{mx_min},{my_min},{mx_max},{my_max}",
            "bboxSR": "3857",
            "size": f"{pixel_dim},{pixel_dim}",
            "imageSR": "3857",
            "format": "png",
            "transparent": "true",
            "f": "image",
        }

        response = requests.get(service_url, params=export_params, timeout=15)
        if response.status_code == 200:
            img = Image.open(BytesIO(response.content)).convert("RGB")
            buffer = BytesIO()
            img.save(buffer, format="JPEG", quality=85, optimize=True)
            return buffer.getvalue()
    except Exception:
        pass
    return None

def main():
    print(f"Generating 200m grid for bounding box {TOP_LEFT} -> {BOTTOM_RIGHT}...")
    grid_points = generate_extent_grid(TOP_LEFT, BOTTOM_RIGHT, step_meters=TILE_SIZE_METERS)
    print(f"Total tiles to download: {len(grid_points)}")

    schema = pa.schema([
        pa.field("tile_id", pa.int64()),
        pa.field("lat", pa.float32()),
        pa.field("lon", pa.float32()),
        pa.field("image_bytes", pa.binary()),
    ])

    rows = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_DOWNLOAD_THREADS) as executor:
        futures = {
            executor.submit(download_point_image, pt[0], pt[1], TILE_SIZE_METERS, IMAGE_PIXEL_DIM): (idx, pt)
            for idx, pt in enumerate(grid_points)
        }
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Fetching Satellite Imagery"):
            idx, (lon, lat) = futures[future]
            img_bytes = future.result()
            if img_bytes:
                rows.append({
                    "tile_id": idx,
                    "lat": float(lat),
                    "lon": float(lon),
                    "image_bytes": img_bytes,
                })

    print(f"Saving {len(rows)} valid image tiles to {OUTPUT_PARQUET}...")
    df = pd.DataFrame(rows)
    table = pa.Table.from_pandas(df, schema=schema)
    pq.write_table(table, OUTPUT_PARQUET, compression="ZSTD")
    print("Done extracting extent tiles.")

if __name__ == "__main__":
    main()

Generating 200m grid for bounding box (33.907267674195495, -118.37438955894808) -> (33.875205419908916, -118.28886560981613)...
Total tiles to download: 4085


Fetching Satellite Imagery: 100%|██████████| 4085/4085 [03:06<00:00, 21.88it/s]


Saving 4030 valid image tiles to extent_tiles.parquet...
Done extracting extent tiles.


In [5]:
import base64
from io import BytesIO
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image
import torch
import open_clip
from tqdm import tqdm

# Optional: Try importing ruptures for robust change-point detection
try:
    import ruptures as rpt
    HAS_RUPTURES = True
except ImportError:
    HAS_RUPTURES = False

# --- Configuration ---
INPUT_PARQUET = "extent_tiles.parquet"
OUTPUT_HTML = "clip_search_results.html"
TOP_K = 10  # Number of top results considered per query

MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

QUERIES = [
    "baseball field",
    "solar panels on roof or field",
    "shipping containers or port Terminal",
    "swimming pool in residential area",
    "dense traffic intersection",
    "industrial warehouse complex",
    "marina with docked boats",
    "tennis courts",
    "parking lot full of cars",
    "construction site with bare earth",
]

# --- Post-Processing: 1D Change Point Detection ---
def find_first_changepoint(scores: np.ndarray) -> int:
    """
    Finds the index of the first major drop / change point in a 1D array of sorted scores.
    Returns the threshold index k, where items [0:k] are considered 'Relevant'.
    """
    if len(scores) <= 2:
        return len(scores)

    # Method 1: Advanced detection using Ruptures library (PELT / L2 cost) if installed
    if HAS_RUPTURES:
        try:
            algo = rpt.Pelt(model="l2", min_size=2).fit(scores)
            bkps = algo.predict(pen=1.5)
            if len(bkps) > 0 and bkps[0] < len(scores):
                return int(bkps[0])
        except Exception:
            pass

    # Method 2: Maximum drop / 1D derivative peak detection
    diffs = np.abs(np.diff(scores))
    
    # Penalize later indices slightly to favor early drops in highly sorted lists
    decay = np.exp(-np.linspace(0, 2, len(diffs)))
    weighted_diffs = diffs * decay
    
    changepoint_idx = int(np.argmax(weighted_diffs)) + 1
    
    # Safeguard bounds (at least 1, max top_k)
    return max(1, min(changepoint_idx, len(scores)))

def load_data(parquet_path):
    print(f"Loading image parquet from {parquet_path}...")
    table = pq.read_table(parquet_path)
    return table.to_pandas()

def generate_image_embeddings(df, model, preprocess, device, batch_size=64):
    print("Encoding satellite images into CLIP feature embeddings...")
    embeddings = []
    images_raw = df["image_bytes"].tolist()
    
    for i in tqdm(range(0, len(images_raw), batch_size), desc="Image Embedding"):
        batch_bytes = images_raw[i:i+batch_size]
        pil_images = [Image.open(BytesIO(b)).convert("RGB") for b in batch_bytes]
        image_tensors = torch.stack([preprocess(img) for img in pil_images]).to(device)
        
        with torch.no_grad(), torch.cuda.amp.autocast():
            image_features = model.encode_image(image_tensors)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            embeddings.append(image_features.cpu().numpy())
            
    return np.vstack(embeddings)

def generate_text_embeddings(queries, model, tokenizer, device):
    print("Encoding text queries into CLIP feature embeddings...")
    text_tokens = tokenizer(queries).to(device)
    with torch.no_grad(), torch.cuda.amp.autocast():
        text_features = model.encode_text(text_tokens)
        text_features /= text_features.norm(dim=-1, keepdim=True)
    return text_features.cpu().numpy()

def render_html_report(df, queries, similarities, top_k=50):
    print("Generating HTML gallery report with Change-Point Highlighting & Popups...")
    
    html_sections = []
    nav_links = "".join([f'<a href="#q-{i}">{q}</a>' for i, q in enumerate(queries)])
    
    for q_idx, query_text in enumerate(queries):
        sim_scores = similarities[q_idx]
        top_indices = np.argsort(sim_scores)[::-1][:top_k]
        top_scores = sim_scores[top_indices]
        
        # --- Run Change-Point Postprocessing ---
        cp_idx = find_first_changepoint(top_scores)
        
        grid_cards = []
        for rank, idx in enumerate(top_indices):
            score = sim_scores[idx]
            row = df.iloc[idx]
            is_relevant = rank < cp_idx
            
            b64_img = base64.b64encode(row["image_bytes"]).decode("utf-8")
            img_src = f"data:image/jpeg;base64,{b64_img}"
            maps_url = f"https://www.google.com/maps?q={row['lat']},{row['lon']}"
            
            card_class = "card relevant" if is_relevant else "card"
            badge = '<span class="badge">RELEVANT</span>' if is_relevant else ''
            
            card_html = f"""
            <div class="{card_class}" 
                 onclick="openModal('{img_src}', '{score:.4f}', '{row['lat']:.5f}', '{row['lon']:.5f}', '{maps_url}', {str(is_relevant).lower()})">
                {badge}
                <img src="{img_src}" alt="Tile Image"/>
                <div class="card-info">
                    <span class="score">Score: {score:.3f}</span>
                    <span class="coords">{row['lat']:.4f}, {row['lon']:.4f}</span>
                </div>
            </div>
            """
            grid_cards.append(card_html)
            
        section_html = f"""
        <section id="q-{q_idx}" class="query-section">
            <div class="query-header">
                <h2>Query {q_idx+1}: <span class="query-title">"{query_text}"</span></h2>
                <div class="cp-info">Detected Change-Point @ Rank <strong>{cp_idx}</strong> (Score Cutoff: {top_scores[cp_idx-1]:.3f})</div>
            </div>
            <div class="grid">
                {''.join(grid_cards)}
            </div>
        </section>
        """
        html_sections.append(section_html)

    full_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>CLIP Spatial Search + Change-Point Detection</title>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; background: #0b0f19; color: #f8fafc; margin: 0; padding: 20px; }}
        header {{ background: #1e293b; padding: 16px 20px; border-radius: 12px; margin-bottom: 20px; position: sticky; top: 10px; z-index: 100; box-shadow: 0 4px 20px rgba(0,0,0,0.6); }}
        h1 {{ margin: 0 0 10px 0; font-size: 1.4rem; }}
        .nav {{ display: flex; flex-wrap: wrap; gap: 8px; max-height: 90px; overflow-y: auto; }}
        .nav a {{ background: #334155; color: #38bdf8; padding: 4px 10px; border-radius: 6px; text-decoration: none; font-size: 0.8rem; transition: 0.2s; }}
        .nav a:hover {{ background: #0284c7; color: #fff; }}
        
        .query-section {{ margin-bottom: 40px; background: #131c2e; padding: 20px; border-radius: 12px; border: 1px solid #1e293b; }}
        .query-header {{ display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #334155; padding-bottom: 10px; flex-wrap: wrap; gap: 10px; }}
        h2 {{ margin: 0; font-size: 1.15rem; color: #94a3b8; }}
        .query-title {{ color: #38bdf8; }}
        .cp-info {{ background: #0f172a; border: 1px solid #3b82f6; color: #60a5fa; padding: 4px 12px; border-radius: 20px; font-size: 0.8rem; }}
        
        .grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(150px, 1fr)); gap: 12px; margin-top: 15px; }}
        
        /* Card & Relevance Styling */
        .card {{ background: #0f172a; border-radius: 8px; overflow: hidden; border: 2px solid #1e293b; text-align: center; display: flex; flex-direction: column; cursor: pointer; position: relative; transition: transform 0.15s, border-color 0.2s; opacity: 0.6; }}
        .card:hover {{ transform: translateY(-3px); opacity: 1.0; }}
        .card img {{ width: 100%; height: 140px; object-fit: cover; display: block; }}
        
        .card.relevant {{ border-color: #10b981; box-shadow: 0 0 10px rgba(16, 185, 129, 0.25); opacity: 1.0; }}
        .card.relevant:hover {{ box-shadow: 0 0 16px rgba(16, 185, 129, 0.5); }}
        
        .badge {{ position: absolute; top: 6px; left: 6px; background: #10b981; color: #000; font-weight: bold; font-size: 0.65rem; padding: 2px 6px; border-radius: 4px; box-shadow: 0 2px 4px rgba(0,0,0,0.5); }}
        
        .card-info {{ padding: 6px; display: flex; flex-direction: column; gap: 2px; font-size: 0.75rem; }}
        .score {{ font-weight: bold; color: #10b981; }}
        .coords {{ color: #64748b; }}

        /* Modal Overlay Styling */
        .modal-overlay {{ display: none; position: fixed; top: 0; left: 0; width: 100%; height: 100%; background: rgba(0, 0, 0, 0.85); backdrop-filter: blur(5px); z-index: 1000; justify-content: center; align-items: center; }}
        .modal-content {{ background: #1e293b; padding: 20px; border-radius: 12px; max-width: 450px; width: 90%; text-align: center; border: 1px solid #334155; position: relative; box-shadow: 0 10px 30px rgba(0,0,0,0.8); }}
        .modal-content img {{ width: 100%; height: 320px; object-fit: cover; border-radius: 8px; border: 1px solid #475569; }}
        .close-btn {{ position: absolute; top: 10px; right: 14px; font-size: 1.5rem; color: #94a3b8; cursor: pointer; }}
        .close-btn:hover {{ color: #fff; }}
        .modal-details {{ margin-top: 15px; display: flex; flex-direction: column; gap: 8px; font-size: 0.9rem; }}
        .modal-link {{ display: inline-block; margin-top: 10px; background: #0284c7; color: #fff; padding: 8px 16px; border-radius: 6px; text-decoration: none; font-weight: bold; }}
        .modal-link:hover {{ background: #0369a1; }}
    </style>
</head>
<body>

    <header>
        <h1>CLIP Spatial Search + Post-Processing Dashboard</h1>
        <div class="nav">{nav_links}</div>
    </header>

    {''.join(html_sections)}

    <!-- Lightbox Popup Modal -->
    <div id="imageModal" class="modal-overlay" onclick="closeModal(event)">
        <div class="modal-content" onclick="event.stopPropagation()">
            <span class="close-btn" onclick="closeModal(null)">&times;</span>
            <img id="modalImg" src="" alt="Full Preview"/>
            <div class="modal-details">
                <div id="modalScore"></div>
                <div id="modalStatus"></div>
                <a id="modalMapLink" class="modal-link" href="" target="_blank">Open location in Google Maps</a>
            </div>
        </div>
    </div>

    <script>
        function openModal(imgSrc, score, lat, lon, mapsUrl, isRelevant) {{
            document.getElementById('modalImg').src = imgSrc;
            document.getElementById('modalScore').innerHTML = '<strong>Similarity Score:</strong> ' + score;
            document.getElementById('modalStatus').innerHTML = '<strong>Classification:</strong> ' + 
                (isRelevant ? '<span style="color:#10b981;font-weight:bold;">RELEVANT</span>' : '<span style="color:#94a3b8;">IRRELEVANT / BELOW CUTOFF</span>');
            document.getElementById('modalMapLink').href = mapsUrl;
            document.getElementById('imageModal').style.display = 'flex';
        }}

        function closeModal(event) {{
            if (!event || event.target.id === 'imageModal' || event.target.className === 'close-btn') {{
                document.getElementById('imageModal').style.display = 'none';
            }}
        }}

        document.addEventListener('keydown', function(event) {{
            if (event.key === "Escape") {{
                closeModal(null);
            }}
        }});
    </script>
</body>
</html>
"""
    with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
        f.write(full_html)
    print(f"Interactive dashboard with change-point highlights and modals exported to: {OUTPUT_HTML}")

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Executing pipeline on device: {device}")
    
    model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED, device=device)
    tokenizer = open_clip.get_tokenizer(MODEL_NAME)
    model.eval()

    df = load_data(INPUT_PARQUET)
    image_embeddings = generate_image_embeddings(df, model, preprocess, device)
    text_embeddings = generate_text_embeddings(QUERIES, model, tokenizer, device)

    similarities = np.dot(text_embeddings, image_embeddings.T)

    render_html_report(df, QUERIES, similarities, top_k=TOP_K)

if __name__ == "__main__":
    main()

Executing pipeline on device: cuda
Loading image parquet from extent_tiles.parquet...
Encoding satellite images into CLIP feature embeddings...


Image Embedding:   0%|          | 0/15 [00:00<?, ?it/s]C:\Users\sus14836\AppData\Local\Temp\ipykernel_26476\939106662.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():
Image Embedding: 100%|██████████| 15/15 [00:02<00:00,  6.93it/s]
C:\Users\sus14836\AppData\Local\Temp\ipykernel_26476\939106662.py:95: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


Encoding text queries into CLIP feature embeddings...
Generating HTML gallery report with Change-Point Highlighting & Popups...
Interactive dashboard with change-point highlights and modals exported to: clip_search_results.html


In [8]:
import base64
from io import BytesIO
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image
import torch
import open_clip
from tqdm import tqdm
from scipy.signal import savgol_filter
from sklearn.cluster import DBSCAN

# Try using kneed library for maximum curvature; fallback to internal geometry if absent
try:
    from kneed import KneeLocator
    HAS_KNEED = True
except ImportError:
    HAS_KNEED = False

# --- Configuration ---
INPUT_PARQUET = "extent_tiles.parquet"
OUTPUT_HTML = "clip_search_results.html"
TOP_K_CANDIDATES = 100  # Number of initial candidates evaluated for change point

MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

QUERIES = [
    # --- Original Queries ---
    "baseball field",
    "solar panels on roof or field",
    "shipping containers or port Terminal",
    "swimming pool in residential area",
    "dense traffic intersection",
    "industrial warehouse complex",
    "marina with docked boats",
    "tennis courts",
    "parking lot full of cars",
    "construction site with bare earth",

    # --- Sports & Recreation ---
    "football or soccer stadium with field markings",
    "golf course fairways and sand traps",
    "athletics running track oval",
    "outdoor basketball courts",
    "skate park with concrete ramps",
    "race track or motorsport circuit",
    "amusement park with roller coasters",
    "public park with green trees and walking paths",

    # --- Urban Infrastructure & Residential ---
    "suburban neighborhood with grid pattern houses",
    "dense high-rise apartment complex",
    "cul-de-sac suburban road layout",
    "mobile home park or trailer park",
    "cemetery with grid of headstones",
    "shopping mall complex with large parking area",
    "gas station or truck stop",
    "school ground with playground and field",
    "hospital complex with helipad",
    "urban plaza or pedestrian square",

    # --- Transportation & Logistics ---
    "airport runway and tarmac with parked airplanes",
    "railroad railyard with freight trains and tracks",
    "highway cloverleaf interchange",
    "bridge crossing over river or water",
    "roundabout or traffic circle",
    "bus depot or truck fleet terminal",
    "toll plaza on highway",
    "pier or jetty extending into ocean",
    "canal or artificial waterway with locks",
    "dry dock for ship maintenance",

    # --- Industrial & Energy Infrastructure ---
    "oil or gas storage tanks",
    "wind turbines in field or offshore",
    "coal power plant or cooling towers",
    "water treatment plant with circular settling tanks",
    "electrical substation with transformers and pylons",
    "mining pit or open pit quarry",
    "oil refinery with complex pipelines",
    "solar farm with rows of PV panels",
    "recycling yard or scrap yard with metal heaps",
    "lumber yard with stacked timber",

    # --- Agriculture & Land Use ---
    "center pivot circular irrigation fields",
    "terraced farming fields on hillside",
    "greenhouses or polytunnels in agricultural area",
    "orchard with neat rows of fruit trees",
    "vineyard with parallel rows of vines",
    "crop fields with rectangular boundaries",
    "hay bales in harvested field",
    "dairy farm or livestock feedlot",
    "fish farm or aquaculture ponds",
    "paddy fields flooded with water",

    # --- Natural & Physical Geography ---
    "meandering river through valley",
    "sand dunes in desert",
    "volcanic crater or caldera",
    "glacier or ice sheet",
    "coastal beach with ocean waves breaking",
    "coral reef under clear shallow ocean water",
    "forest canopy with dense tree cover",
    "mangrove swamp or coastal wetland",
    "mountain peak with snow cover",
    "river delta flowing into sea",
    "cliff face along ocean coastline",
    "lake or reservoir with dam structure",

    # --- Environmental & Land Disturbances ---
    "wildfire burn scar or scorched forest",
    "deforestation or clear-cut timber forest",
    "flooded agricultural fields or urban area",
    "landslide or mudslide scar",
    "coastal erosion or shrinking beach",
    "drying lake bed with salt flats",
    "smokestacks emitting plume or smoke",

    # --- Specialized & Rare Features ---
    "helipad on building rooftop",
    "archaeological ruins or ancient structures",
    "military base with barracks and airfield",
    "drive-in theater screen and parking",
    "solar evaporative salt pans with bright colors",
    "log boom or timber floating on river",
    "fountain or water display in park",
    "footbridge over highway",
    "sewage lagoon or oxidation pond",
    "radio or satellite communication dish antenna array",

    # --- Pattern & Texture Specifics ---
    "neat geometric grid of urban streets",
    "curved winding mountain road",
    "bright red roofed buildings",
    "bright blue roofs or tarps",
    "dense cluster of small boats",
    "checkerboard pattern of agricultural fields",
    "shadows cast by tall skyscrapers",
    "muddy brown water meeting clear blue ocean water",
    "zigzag mountain road switchbacks",
    "patches of bare soil mixed with forest"
]

# --- Helper: Coordinate Transformation ---
def wgs84_to_web_mercator(lon, lat):
    lon_rad = np.radians(lon)
    x = 6378137.0 * lon_rad
    y = 6378137.0 * np.log(np.tan(np.pi / 4.0 + np.radians(lat) / 2.0))
    return x, y

# --- 1. Knee Detection (Kneedle / Max Curvature) ---
def detect_knee_point(scores: np.ndarray) -> int:
    """
    Detects the knee point in sorted similarity scores using:
    1. Kneedle algorithm (Distance from straight line)
    2. Fallback to Largest Gap (1st derivative)
    3. Fallback to Z-score (>2.5 std devs)
    """
    n = len(scores)
    if n <= 3:
        return n

    # Step A: Light smoothing to prevent micro-jitter noise
    if n >= 7:
        smooth_scores = savgol_filter(scores, window_length=min(11, n if n % 2 != 0 else n - 1), polyorder=2)
    else:
        smooth_scores = scores.copy()

    # Strategy 1: Kneedle Algorithm
    if HAS_KNEED:
        x_vals = np.arange(n)
        kl = KneeLocator(x_vals, smooth_scores, curve="convex", direction="decreasing", S=1.0)
        if kl.knee is not None and 1 <= kl.knee < n - 1:
            return int(kl.knee) + 1

    # Strategy 1 Sub-fallback: Manual perpendicular distance to chord line
    x_norm = np.linspace(0, 1, n)
    y_norm = (smooth_scores - smooth_scores[-1]) / (smooth_scores[0] - smooth_scores[-1] + 1e-8)
    
    # Distance from line connecting (0, y_norm[0]) to (1, y_norm[-1])
    line_vec = np.array([1.0, y_norm[-1] - y_norm[0]])
    line_vec /= np.linalg.norm(line_vec)
    
    vec_to_points = np.column_stack([x_norm, y_norm - y_norm[0]])
    distances = np.abs(np.cross(line_vec, vec_to_points))
    max_dist_idx = int(np.argmax(distances))
    
    # Accept max distance if it forms a genuine elbow drop
    if distances[max_dist_idx] > 0.05 and 1 <= max_dist_idx < n - 1:
        return max_dist_idx + 1

    # Strategy 2: Largest Gap (\Delta s)
    diffs = np.abs(np.diff(smooth_scores))
    max_gap_idx = int(np.argmax(diffs)) + 1
    if diffs[max_gap_idx - 1] > 0.02:
        return max_gap_idx

    # Strategy 3: Dynamic Z-score Fallback relative to baseline
    mean_s = np.mean(scores)
    std_s = np.std(scores) + 1e-6
    z_scores = (scores - mean_s) / std_s
    z_hits = np.where(z_scores >= 2.0)[0]
    
    if len(z_hits) > 0:
        return int(z_hits[-1]) + 1

    return min(15, n)  # Default safety fallback

# --- 2. Spatial Consistency Filter ---
def filter_spatial_isolated_tiles(relevant_df: pd.DataFrame, max_dist_meters: float = 350.0) -> np.ndarray:
    """
    Uses DBSCAN in Web Mercator metric projection to keep spatial clusters 
    and remove isolated single-tile false positives.
    """
    if len(relevant_df) <= 2:
        return np.ones(len(relevant_df), dtype=bool)

    coords = np.array([
        wgs84_to_web_mercator(lon, lat) 
        for lon, lat in zip(relevant_df["lon"], relevant_df["lat"])
    ])

    # DBSCAN with eps = 350 meters (~1.5 tile lengths distance)
    db = DBSCAN(eps=max_dist_meters, min_samples=1).fit(coords)
    labels = db.labels_

    # Count cluster sizes
    unique_labels, counts = np.unique(labels, return_counts=True)
    cluster_sizes = dict(zip(unique_labels, counts))

    # Keep items if they are part of a multi-tile cluster, OR if top score candidate
    is_spatially_valid = np.array([cluster_sizes[lbl] >= 2 or idx == 0 for idx, lbl in enumerate(labels)])
    return is_spatially_valid

# --- Embedding & Inference Pipeline ---
def load_data(parquet_path):
    print(f"Loading image parquet from {parquet_path}...")
    table = pq.read_table(parquet_path)
    return table.to_pandas()

def generate_image_embeddings(df, model, preprocess, device, batch_size=64):
    print("Encoding satellite images into CLIP feature embeddings...")
    embeddings = []
    images_raw = df["image_bytes"].tolist()
    
    for i in tqdm(range(0, len(images_raw), batch_size), desc="Image Embedding"):
        batch_bytes = images_raw[i:i+batch_size]
        pil_images = [Image.open(BytesIO(b)).convert("RGB") for b in batch_bytes]
        image_tensors = torch.stack([preprocess(img) for img in pil_images]).to(device)
        
        with torch.no_grad(), torch.cuda.amp.autocast():
            image_features = model.encode_image(image_tensors)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            embeddings.append(image_features.cpu().numpy())
            
    return np.vstack(embeddings)

def generate_text_embeddings(queries, model, tokenizer, device):
    print("Encoding text queries into CLIP feature embeddings...")
    text_tokens = tokenizer(queries).to(device)
    with torch.no_grad(), torch.cuda.amp.autocast():
        text_features = model.encode_text(text_tokens)
        text_features /= text_features.norm(dim=-1, keepdim=True)
    return text_features.cpu().numpy()

# --- HTML Generator ---
def render_html_report(df, queries, similarities, top_k=100):
    print("Generating HTML gallery dashboard with interactive filtering modes...")
    
    html_sections = []
    nav_links = "".join([f'<a href="#q-{i}">{q}</a>' for i, q in enumerate(queries)])
    
    for q_idx, query_text in enumerate(queries):
        sim_scores = similarities[q_idx]
        top_indices = np.argsort(sim_scores)[::-1][:top_k]
        top_scores = sim_scores[top_indices]
        
        # 1. Detect Change Point
        knee_idx = detect_knee_point(top_scores)
        
        # 2. Run Spatial Consistency
        cand_df = df.iloc[top_indices[:knee_idx]].copy()
        spatial_mask = filter_spatial_isolated_tiles(cand_df)
        
        grid_cards = []
        for rank, idx in enumerate(top_indices):
            score = float(sim_scores[idx])
            row = df.iloc[idx]
            
            is_above_knee = rank < knee_idx
            is_spatially_valid = is_above_knee and spatial_mask[rank]
            
            b64_img = base64.b64encode(row["image_bytes"]).decode("utf-8")
            img_src = f"data:image/jpeg;base64,{b64_img}"
            maps_url = f"https://www.google.com/maps?q={row['lat']},{row['lon']}"
            
            card_html = f"""
            <div class="card" 
                 data-rank="{rank+1}" 
                 data-score="{score:.4f}" 
                 data-knee="{'true' if is_above_knee else 'false'}"
                 data-auto="{'true' if is_spatially_valid else 'false'}"
                 onclick="openModal('{img_src}', '{score:.4f}', '{row['lat']:.5f}', '{row['lon']:.5f}', '{maps_url}', {str(is_spatially_valid).lower()})">
                <span class="badge auto-badge">AUTO MATCH</span>
                <span class="badge knee-badge">KNEE CUTOFF</span>
                <img src="{img_src}" alt="Tile Image"/>
                <div class="card-info">
                    <span class="score">Score: {score:.3f}</span>
                    <span class="coords">{row['lat']:.4f}, {row['lon']:.4f}</span>
                </div>
            </div>
            """
            grid_cards.append(card_html)
            
        section_html = f"""
        <section id="q-{q_idx}" class="query-section">
            <div class="query-header">
                <h2>Query {q_idx+1}: <span class="query-title">"{query_text}"</span></h2>
                <div class="cp-info">
                    <span>Knee Rank: <strong>{knee_idx}</strong> (Score: {top_scores[knee_idx-1]:.3f})</span>
                    <span>Spatial Filter Passed: <strong>{np.sum(spatial_mask)}</strong> tiles</span>
                </div>
            </div>
            <div class="grid">
                {''.join(grid_cards)}
            </div>
        </section>
        """
        html_sections.append(section_html)

    full_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>CLIP Spatial Engine Dashboard</title>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; background: #0b0f19; color: #f8fafc; margin: 0; padding: 20px; }}
        header {{ background: #1e293b; padding: 16px 20px; border-radius: 12px; margin-bottom: 20px; position: sticky; top: 10px; z-index: 100; box-shadow: 0 4px 20px rgba(0,0,0,0.6); }}
        .header-top {{ display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 15px; margin-bottom: 12px; }}
        h1 {{ margin: 0; font-size: 1.4rem; }}
        
        /* Control Panel */
        .controls {{ display: flex; align-items: center; gap: 15px; background: #0f172a; padding: 8px 16px; border-radius: 8px; border: 1px solid #334155; }}
        .controls label {{ font-size: 0.85rem; color: #94a3b8; font-weight: bold; }}
        .controls select, .controls input {{ background: #1e293b; color: #38bdf8; border: 1px solid #475569; padding: 4px 8px; border-radius: 6px; font-size: 0.85rem; outline: none; }}
        
        .nav {{ display: flex; flex-wrap: wrap; gap: 8px; max-height: 80px; overflow-y: auto; }}
        .nav a {{ background: #334155; color: #38bdf8; padding: 4px 10px; border-radius: 6px; text-decoration: none; font-size: 0.8rem; transition: 0.2s; }}
        .nav a:hover {{ background: #0284c7; color: #fff; }}
        
        .query-section {{ margin-bottom: 40px; background: #131c2e; padding: 20px; border-radius: 12px; border: 1px solid #1e293b; }}
        .query-header {{ display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #334155; padding-bottom: 10px; flex-wrap: wrap; gap: 10px; }}
        h2 {{ margin: 0; font-size: 1.15rem; color: #94a3b8; }}
        .query-title {{ color: #38bdf8; }}
        .cp-info {{ display: flex; gap: 12px; font-size: 0.8rem; background: #0f172a; padding: 4px 12px; border-radius: 20px; border: 1px solid #3b82f6; color: #60a5fa; }}
        
        .grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(150px, 1fr)); gap: 12px; margin-top: 15px; }}
        
        /* Card Styling & Modes */
        .card {{ background: #0f172a; border-radius: 8px; overflow: hidden; border: 2px solid #1e293b; text-align: center; display: flex; flex-direction: column; cursor: pointer; position: relative; transition: transform 0.15s, border-color 0.2s, opacity 0.2s; opacity: 0.35; }}
        .card:hover {{ transform: translateY(-3px); opacity: 1.0 !important; }}
        .card img {{ width: 100%; height: 140px; object-fit: cover; display: block; }}
        
        .badge {{ position: absolute; top: 6px; left: 6px; color: #000; font-weight: bold; font-size: 0.65rem; padding: 2px 6px; border-radius: 4px; box-shadow: 0 2px 4px rgba(0,0,0,0.5); display: none; }}
        .auto-badge {{ background: #10b981; }}
        .knee-badge {{ background: #f59e0b; }}

        /* Active highlight states driven by interactive mode dropdown */
        body[data-mode="auto"] .card[data-auto="true"] {{ border-color: #10b981; box-shadow: 0 0 10px rgba(16, 185, 129, 0.3); opacity: 1.0; }}
        body[data-mode="auto"] .card[data-auto="true"] .auto-badge {{ display: block; }}

        body[data-mode="knee"] .card[data-knee="true"] {{ border-color: #f59e0b; box-shadow: 0 0 10px rgba(245, 158, 11, 0.3); opacity: 1.0; }}
        body[data-mode="knee"] .card[data-knee="true"] .knee-badge {{ display: block; }}

        body[data-mode="topk"] .card.topk-active {{ border-color: #3b82f6; opacity: 1.0; }}
        body[data-mode="thresh"] .card.thresh-active {{ border-color: #ec4899; opacity: 1.0; }}

        .card-info {{ padding: 6px; display: flex; flex-direction: column; gap: 2px; font-size: 0.75rem; }}
        .score {{ font-weight: bold; color: #10b981; }}
        .coords {{ color: #64748b; }}

        /* Lightbox Modal */
        .modal-overlay {{ display: none; position: fixed; top: 0; left: 0; width: 100%; height: 100%; background: rgba(0, 0, 0, 0.85); backdrop-filter: blur(5px); z-index: 1000; justify-content: center; align-items: center; }}
        .modal-content {{ background: #1e293b; padding: 20px; border-radius: 12px; max-width: 450px; width: 90%; text-align: center; border: 1px solid #334155; position: relative; box-shadow: 0 10px 30px rgba(0,0,0,0.8); }}
        .modal-content img {{ width: 100%; height: 320px; object-fit: cover; border-radius: 8px; border: 1px solid #475569; }}
        .close-btn {{ position: absolute; top: 10px; right: 14px; font-size: 1.5rem; color: #94a3b8; cursor: pointer; }}
        .close-btn:hover {{ color: #fff; }}
        .modal-details {{ margin-top: 15px; display: flex; flex-direction: column; gap: 8px; font-size: 0.9rem; }}
        .modal-link {{ display: inline-block; margin-top: 10px; background: #0284c7; color: #fff; padding: 8px 16px; border-radius: 6px; text-decoration: none; font-weight: bold; }}
        .modal-link:hover {{ background: #0369a1; }}
    </style>
</head>
<body data-mode="auto">

    <header>
        <div class="header-top">
            <h1>CLIP Spatial Search Engine</h1>
            <div class="controls">
                <label for="modeSelect">Detection Mode:</label>
                <select id="modeSelect" onchange="updateMode()">
                    <option value="auto" selected>Auto (Knee + Spatial Filter)</option>
                    <option value="knee">Raw Knee (Kneedle Cutoff)</option>
                    <option value="topk">Top-K Cutoff</option>
                    <option value="thresh">Raw Threshold</option>
                </select>
                
                <span id="topkBox" style="display:none;">
                    <label>K:</label>
                    <input type="number" id="topkInput" value="20" min="1" max="100" style="width: 50px;" oninput="applyCustomFilters()"/>
                </span>

                <span id="threshBox" style="display:none;">
                    <label>Min Similarity:</label>
                    <input type="number" id="threshInput" value="0.25" step="0.01" min="0" max="1" style="width: 60px;" oninput="applyCustomFilters()"/>
                </span>
            </div>
        </div>
        <div class="nav">{nav_links}</div>
    </header>

    {''.join(html_sections)}

    <!-- Lightbox Modal -->
    <div id="imageModal" class="modal-overlay" onclick="closeModal(event)">
        <div class="modal-content" onclick="event.stopPropagation()">
            <span class="close-btn" onclick="closeModal(null)">&times;</span>
            <img id="modalImg" src="" alt="Full Preview"/>
            <div class="modal-details">
                <div id="modalScore"></div>
                <div id="modalStatus"></div>
                <a id="modalMapLink" class="modal-link" href="" target="_blank">Open location in Google Maps</a>
            </div>
        </div>
    </div>

    <script>
        function updateMode() {{
            const mode = document.getElementById('modeSelect').value;
            document.body.setAttribute('data-mode', mode);
            
            document.getElementById('topkBox').style.display = (mode === 'topk') ? 'inline-block' : 'none';
            document.getElementById('threshBox').style.display = (mode === 'thresh') ? 'inline-block' : 'none';
            
            applyCustomFilters();
        }}

        function applyCustomFilters() {{
            const mode = document.body.getAttribute('data-mode');
            const kVal = parseInt(document.getElementById('topkInput').value) || 20;
            const threshVal = parseFloat(document.getElementById('threshInput').value) || 0.25;

            document.querySelectorAll('.card').forEach(card => {{
                const rank = parseInt(card.getAttribute('data-rank'));
                const score = parseFloat(card.getAttribute('data-score'));

                card.classList.toggle('topk-active', rank <= kVal);
                card.classList.toggle('thresh-active', score >= threshVal);
            }});
        }}

        function openModal(imgSrc, score, lat, lon, mapsUrl, isAutoMatch) {{
            document.getElementById('modalImg').src = imgSrc;
            document.getElementById('modalScore').innerHTML = '<strong>Similarity Score:</strong> ' + score;
            document.getElementById('modalStatus').innerHTML = '<strong>Classification:</strong> ' + 
                (isAutoMatch ? '<span style="color:#10b981;font-weight:bold;">AUTO MATCH (Knee + Spatial Passed)</span>' : '<span style="color:#94a3b8;">BELOW AUTO CUTOFF</span>');
            document.getElementById('modalMapLink').href = mapsUrl;
            document.getElementById('imageModal').style.display = 'flex';
        }}

        function closeModal(event) {{
            if (!event || event.target.id === 'imageModal' || event.target.className === 'close-btn') {{
                document.getElementById('imageModal').style.display = 'none';
            }}
        }}

        document.addEventListener('keydown', function(event) {{
            if (event.key === "Escape") closeModal(null);
        }});

        updateMode();
    </script>
</body>
</html>
"""
    with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
        f.write(full_html)
    print(f"Complete pipeline dashboard exported to: {OUTPUT_HTML}")

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Executing pipeline on device: {device}")
    
    model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED, device=device)
    tokenizer = open_clip.get_tokenizer(MODEL_NAME)
    model.eval()

    df = load_data(INPUT_PARQUET)
    image_embeddings = generate_image_embeddings(df, model, preprocess, device)
    text_embeddings = generate_text_embeddings(QUERIES, model, tokenizer, device)

    similarities = np.dot(text_embeddings, image_embeddings.T)

    render_html_report(df, QUERIES, similarities, top_k=TOP_K_CANDIDATES)

if __name__ == "__main__":
    main()

Executing pipeline on device: cuda
Loading image parquet from extent_tiles.parquet...
Encoding satellite images into CLIP feature embeddings...


Image Embedding:   0%|          | 0/63 [00:00<?, ?it/s]C:\Users\sus14836\AppData\Local\Temp\ipykernel_26476\2752339693.py:254: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():
Image Embedding: 100%|██████████| 63/63 [00:08<00:00,  7.37it/s]
C:\Users\sus14836\AppData\Local\Temp\ipykernel_26476\2752339693.py:264: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


Encoding text queries into CLIP feature embeddings...
Generating HTML gallery dashboard with interactive filtering modes...


C:\Users\sus14836\AppData\Local\Temp\ipykernel_26476\2752339693.py:188: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  distances = np.abs(np.cross(line_vec, vec_to_points))
C:\Users\sus14836\AppData\Local\Temp\ipykernel_26476\2752339693.py:188: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  distances = np.abs(np.cross(line_vec, vec_to_points))
C:\Users\sus14836\AppData\Local\Temp\ipykernel_26476\2752339693.py:188: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  distances = np.abs(np.cross(line_vec, vec_to_points))
C:\Users\sus14836\AppData\Local\Temp\ipykernel_26476\2752339693.py:188: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in

Complete pipeline dashboard exported to: clip_search_results.html


In [1]:
import pandas as pd

pd.read_parquet(rf"D:\Code\query-earth\extent_tiles.parquet")

,tile_id,lat,lon,image_bytes
0,4,33.878563,-118.373940,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
1,12,33.884529,-118.373940,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
2,35,33.901680,-118.373940,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
3,38,33.903915,-118.373940,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
4,20,33.890495,-118.373940,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
...,...,...,...,...
4025,4083,33.906151,-118.289497,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
4026,4079,33.903168,-118.289497,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
4027,4084,33.906898,-118.289497,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
4028,4080,33.903915,-118.289497,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...


In [2]:
import onnxruntime_genai as og

# 1. Path to your exported ONNX model folder
model_path = rf"E:\Weights\onnx-models\ministral-3b-onnx-int4"

# 2. Initialize ONNX runtime model and tokenizer
print("Loading model into VRAM...")
model = og.Model(model_path)
tokenizer = og.Tokenizer(model)
tokenizer_stream = tokenizer.create_stream()

# 3. Format input prompt (using Mistral chat template format)
prompt = "Explain how spatial indexing works in 2 short sentences."
formatted_prompt = f"<s>[INST] {prompt} [/INST]"

# 4. Set up generator parameters
params = og.GeneratorParams(model)
params.set_search_options(max_length=512, temperature=0.7, top_p=0.9)
params.input_ids = tokenizer.encode(formatted_prompt)

# 5. Run inference with token streaming
generator = og.Generator(model, params)
print("\nResponse:")

while not generator.is_done():
    generator.generate_next_token()
    new_token = generator.get_next_tokens()[0]
    print(tokenizer_stream.decode(new_token), end="", flush=True)

print("\n\nDone!")

Loading model into VRAM...


RuntimeError: Cuda interface not available: Failed to load library: Error loading "onnxruntime-genai-cuda.dll" which is missing. (Error 126)

In [3]:
from llama_cpp import Llama

llm = Llama(
    model_path="ministral-3b-instruct-q5_k_m.gguf",
    n_ctx=4096,
    n_threads=8,
)

response = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": question}
    ]
)

ModuleNotFoundError: No module named 'llama_cpp'

In [ ]:
import hugg

In [ ]:
python -m onnxruntime_genai.models.builder --model ministral --output ministral_onnx

In [ ]:
python -m onnxruntime_genai.models.builder -m ministral -o ministral_onnx -p int4 -e cpu

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("ministral")
model = AutoModelForCausalLM.from_pretrained(
    "ministral",
    torch_dtype="auto"
)

RuntimeError: operator torchvision::nms does not exist

"Age-wise, the community reads as mixed age 5% working-age adults alongside 2% under 18 and 2% 65-plus. Household sizes skew large relative to the national distribution. Rent burden here is typical, and the area is an owner-dominated market. The housing stock is characterized as historic pre-1950. A large share of homes predate 1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Economically, this is an average-income area with moderate levels of poverty. Educational attainment is average in terms of bachelor's degrees, in what is best described as a mixed education profile."

"The population skews mixed age: about 9% are minors and 0% are seniors, with the remainder of working age. Household sizes skew large relative to the national distribution. Housing costs relative to income run typical, in an owner-dominated market. The housing stock is characterized as historic pre1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Economically, this is an average-income area with moderate levels of poverty. Educational attainment is average in terms of bachelor's degrees, in what is best described as a mixed education profile."


"This area has an age profile best described as mixed age, with roughly 2% of residents under 18 and 2% aged 65 or older. Household sizes skew large relative to the national distribution. Housing costs relative to income run typical, in an owner-dominated market. The housing stock is characterized as historic pre1950. A large share of homes predate 1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Economically, this is an average-income area with moderate levels of poverty. Educational attainment is average in terms of bachelor's degrees, in what is best described as a mixed education profile."

{"zip_code": "35", "caption": "Age-wise, the community reads as mixed age \u2014 5% working-age adults alongside 2% under 18 and -2% 65-plus. Household sizes skew large relative to the national distribution. Housing costs relative to income run typical, in an owner-dominated market. The housing stock is characterized as historic pre1950. A large share of homes predate 1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Median household income is average for the country, with moderate poverty rates. The area has a mixed education profile, with average rates of four-year-degree attainment."}


"Age-wise, the community reads as mixed age \u2014 -5% working-age adults alongside -2% under 18 and -2% 65-plus. Household sizes skew large relative to the national distribution. Housing costs relative to income run typical, in an owner-dominated market. The housing stock is characterized as historic pre1950. A large share of homes predate 1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Economically, this is an average-income area with moderate levels of poverty. The area has a mixed education profile, with average rates of four-year-degree attainment."

"This area has an age profile best described as mixed age, with roughly -2% of residents under 18 and -2% aged 65 or older. Household sizes skew large relative to the national distribution. Rent burden here is typical, and the area is an owner-dominated market. The housing stock is characterized as historic pre1950. A large share of homes predate 1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Median household income is average for the country, with moderate poverty rates. The area has a mixed education profile, with average rates of four-year-degree attainment."

"The population skews mixed age: about -2% are minors and -2% are seniors, with the remainder of working age. Household sizes skew large relative to the national distribution. Housing costs relative to income run typical, in an owner-dominated market. The housing stock is characterized as historic pre1950. A large share of homes predate 1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Economically, this is an average-income area with moderate levels of poverty. Educational attainment is average in terms of bachelor's degrees, in what is best described as a mixed education profile."

"Age-wise, the community reads as mixed age \u2014 -5% working-age adults alongside -2% under 18 and -2% 65-plus. Household sizes skew large relative to the national distribution. Housing costs relative to income run typical, in an owner-dominated market. The housing stock is characterized as historic pre1950. A large share of homes predate 1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Median household income is average for the country, with moderate poverty rates. The area has a mixed education profile, with average rates of four-year-degree attainment."
{"zip_code": "6", "caption": "The population skews mixed age: about -2% are minors and -2% are seniors, with the remainder of working age. Household sizes skew large relative to the national distribution. Housing costs relative to income run typical, in an owner-dominated market. The housing stock is characterized as historic pre1950. A large share of homes predate 1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Economically, this is an average-income area with moderate levels of poverty. Educational attainment is average in terms of bachelor's degrees, in what is best described as a mixed education profile."}
{"zip_code": "7", "caption": "This area has an age profile best described as mixed age, with roughly -2% of residents under 18 and -2% aged 65 or older. Household sizes skew large relative to the national distribution. Housing costs relative to income run typical, in an owner-dominated market. The housing stock is characterized as historic pre1950. A large share of homes predate 1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Economically, this is an average-income area with moderate levels of poverty. Educational attainment is average in terms of bachelor's degrees, in what is best described as a mixed education profile."}
{"zip_code": "8", "caption": "This area has an age profile best described as mixed age, with roughly -2% of residents under 18 and -2% aged 65 or older. Household sizes skew large relative to the national distribution. Rent burden here is typical, and the area is an owner-dominated market. The housing stock is characterized as historic pre1950. A large share of homes predate 1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Economically, this is an average-income area with moderate levels of poverty. The area has a mixed education profile, with average rates of four-year-degree attainment."}
{"zip_code": "9", "caption": "The population skews mixed age: about -2% are minors and -2% are seniors, with the remainder of working age. Household sizes skew large relative to the national distribution. Rent burden here is typical, and the area is an owner-dominated market. The housing stock is characterized as historic pre1950. A large share of homes predate 1950. A large share of homes were built after 2000. Neighborhood change indicators suggest a False pattern. Median household income is average for the country, with moderate poverty rates. Educational attainment is average in terms of bachelor's degrees, in what is best described as a mixed education profile."}
